# Adaptive Lifelong IRL for Human-Robot Collaboration

HRI/ICRA-ready implementation with:
- Importance-weighted MaxEnt IRL on demo-augmented MDP
- Adaptive temporal weight decay (interaction_step-based timing)
- Empirically derived overlap threshold (no hardcoded values)
- Baseline comparison: NoReplay, UniformWeight, FixedDecay vs Proposed
- Held-out evaluation to distinguish reward generalisation from memorisation

## Cell 1: Imports and Domain Constants, State Tracker, Recipe Generator

In [1]:
import re, json, random, string, gc, itertools
import numpy as np
from collections import Counter
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
matplotlib.rcParams['figure.dpi'] = 110

RANDOM_SEED   = 42
SHOW_STEPS    = False    # per-step action predictions during online observation
SHOW_CF_STEPS = False    # per-step predictions in forgetting checks
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Kitchen Domain Vocabulary (single source of truth)
CONTAINERS         = ["pot", "pan", "plate", "bowl", "glass", "measuring_cup"]
LIQUID_INGREDIENTS = ["milk", "oil"]
SOLID_INGREDIENTS  = ["tomato", "garlic", "onion", "mushroom", "lettuce",
                       "cheese", "rice", "yoghurt", "strawberries", "banana",
                       "egg", "fish", "chicken", "meat",
                       "salt", "spice1", "spice2", "mixture"]
INGREDIENTS = LIQUID_INGREDIENTS + SOLID_INGREDIENTS
ITEMS       = CONTAINERS + INGREDIENTS
CUTTABLES   = ["tomato", "onion", "mushroom", "lettuce", "banana", "strawberries", "chicken", "fish", "cheese"]
GRATABLE    = ["cheese"]
COOKABLES   = ["meat", "egg", "rice", "tomato", "onion", "mushroom", "chicken", "fish", "mixture"]
SEASONINGS  = ["salt", "spice1", "spice2", "garlic"]
LOCATIONS   = ["storage", "prep_station", "cooking_station", "plating_station", "serving_station", "washing_station", "blending_station"]
TOOLS       = {"prep_station":  ["cutting_board", "grater"],        "cooking_station":  ["stove"],          "washing_station":  ["sink"],       "blending_station":  ["blender"]}








#   The feature_map is built once at class level (not per-instance) to avoid constructing a throwaway object just to read the vocabulary. The feature_map is a dict mapping from "feature name" to column index in the raw state vector. 
#   The feature names are constructed systematically from the vocabulary, so that all possible features are pre-defined and the same across all instances. The feature_map is built on first access and then cached for all future instances.
class StateTracker:
    _CLASS_FEATURE_MAP = None   # built once, shared across all instances
    @classmethod
    def _build_feature_map(cls):
        if cls._CLASS_FEATURE_MAP is not None:      return cls._CLASS_FEATURE_MAP
        fm, idx = {}, 0
        for item in ITEMS:
            for loc in LOCATIONS:                   fm[f"{item}_at_{loc}"] = idx; idx += 1
        for container in CONTAINERS:
            for ingredient in INGREDIENTS:          fm[f"{container}_contains_{ingredient}"] = idx; idx += 1
        for item in CUTTABLES:                      fm[f"{item}_cut"] = idx; idx += 1
        for item in GRATABLE:                       fm[f"{item}_grated"] = idx; idx += 1
        for item in COOKABLES:                      fm[f"{item}_cooked"] = idx; idx += 1
        for item in INGREDIENTS:                    fm[f"{item}_seasoned"] = idx; idx += 1
        for item in ITEMS:                          fm[f"{item}_washed"] = idx; idx += 1
        fm["stove_on"]    = idx; idx += 1
        fm["sink_on"]     = idx; idx += 1
        fm["blender_on"]  = idx; idx += 1
        fm["plate_served"]= idx; idx += 1
        for ingredient in INGREDIENTS:
            if ingredient != "mixture":             fm[f"{ingredient}_in_mixture"] = idx; idx += 1
        for seasoning in SEASONINGS:
            for item in INGREDIENTS:                fm[f"{item}_seasoned_with_{seasoning}"] = idx; idx += 1
        cls._CLASS_FEATURE_MAP = fm
        return fm

    def __init__(self):
        self.feature_map = self.__class__._build_feature_map()
        self.n_features  = len(self.feature_map)
        self.reset()

    def reset(self):
        self.current_state = np.zeros(self.n_features, dtype=int)
        for item in ITEMS:
            if item != "mixture":
                self.set_feature(f"{item}_at_storage", 1)

    def set_feature(self, key, value):
        if key in self.feature_map:
            self.current_state[self.feature_map[key]] = value

    def get_feature(self, key):
        return self.current_state[self.feature_map[key]] if key in self.feature_map else 0

    def get_state_vector(self):
        return self.current_state.copy()

    def get_item_location(self, item):
        for loc in LOCATIONS:
            if self.get_feature(f"{item}_at_{loc}") == 1:
                return loc
        return None

    def is_contained(self, item):
        for container in CONTAINERS:
            if self.get_feature(f"{container}_contains_{item}") == 1:
                return container
        return None

    def apply_action(self, action_str):
        action_str = action_str.strip()
        def _parse(s):
            content = s[s.find("(")+1:s.find(")")]
            return [p.strip() for p in content.split(",")]
        def _loc(part):
            return part.split("=")[1] if "=" in part else part

        if action_str.startswith("transfer"):
            parts    = _parse(action_str)
            item     = parts[0]
            from_loc = _loc(parts[1])
            to_loc   = _loc(parts[2])
            if self.get_feature(f"{item}_at_{from_loc}") != 1:
                raise ValueError(f"Precondition failed: {item} not at {from_loc}")
            if self.is_contained(item):
                raise ValueError(f"Precondition failed: {item} is contained")
            self.set_feature(f"{item}_at_{from_loc}", 0)
            self.set_feature(f"{item}_at_{to_loc}",   1)

        elif action_str.startswith("load"):
            parts     = _parse(action_str)
            item      = parts[0]; container = parts[1]; location = parts[2]
            if self.get_feature(f"{item}_at_{location}") != 1:
                raise ValueError(f"Precondition failed: {item} not at {location}")
            if self.get_feature(f"{container}_at_{location}") != 1:
                raise ValueError(f"Precondition failed: {container} not at {location}")
            if self.is_contained(item):
                raise ValueError(f"Precondition failed: {item} already contained")
            self.set_feature(f"{item}_at_{location}", 0)
            self.set_feature(f"{container}_contains_{item}", 1)

        elif action_str.startswith("unload"):
            parts     = _parse(action_str)
            item      = parts[0]; container = parts[1]; location = parts[2]
            if self.get_feature(f"{container}_contains_{item}") != 1:
                raise ValueError(f"Precondition failed: {item} not in {container}")
            if self.get_feature(f"{container}_at_{location}") != 1:
                raise ValueError(f"Precondition failed: {container} not at {location}")
            self.set_feature(f"{container}_contains_{item}", 0)
            self.set_feature(f"{item}_at_{location}", 1)

        elif action_str.startswith("move_container"):
            parts     = _parse(action_str)
            container = parts[0]; from_loc = _loc(parts[1]); to_loc = _loc(parts[2])
            if self.get_feature(f"{container}_at_{from_loc}") != 1:
                raise ValueError(f"Precondition failed: {container} not at {from_loc}")
            self.set_feature(f"{container}_at_{from_loc}", 0)
            self.set_feature(f"{container}_at_{to_loc}", 1)
            for ingredient in INGREDIENTS:
                if self.get_feature(f"{container}_contains_{ingredient}") == 1:
                    self.set_feature(f"{ingredient}_at_{from_loc}", 0)
                    self.set_feature(f"{ingredient}_at_{to_loc}", 1)

        elif action_str.startswith("cut"):
            parts    = _parse(action_str)
            item     = parts[0]
            location = parts[1] if len(parts) > 1 else "prep_station"
            if item not in CUTTABLES:
                raise ValueError(f"Precondition failed: {item} is not cuttable")
            if self.get_feature(f"{item}_at_{location}") != 1:
                raise ValueError(f"Precondition failed: {item} not at {location}")
            self.set_feature(f"{item}_cut", 1)

        elif action_str.startswith("grate"):
            parts    = _parse(action_str)
            item     = parts[0]
            location = parts[1] if len(parts) > 1 else "prep_station"
            if item not in GRATABLE:
                raise ValueError(f"Precondition failed: {item} is not gratable")
            if self.get_feature(f"{item}_at_{location}") != 1:
                raise ValueError(f"Precondition failed: {item} not at {location}")
            self.set_feature(f"{item}_grated", 1)

        elif action_str.startswith("cook ") or action_str.startswith("cook("):
            parts     = _parse(action_str)
            item      = parts[0]
            container = parts[1] if len(parts) > 1 else "pot"
            location  = parts[2] if len(parts) > 2 else "cooking_station"
            if item not in COOKABLES:
                raise ValueError(f"Precondition failed: {item} is not cookable")
            if self.get_feature(f"{container}_contains_{item}") != 1:
                raise ValueError(f"Precondition failed: {item} not in {container}")
            if self.get_feature(f"{container}_at_{location}") != 1:
                raise ValueError(f"Precondition failed: {container} not at {location}")
            if location == "cooking_station" and self.get_feature("stove_on") != 1:
                raise ValueError("Precondition failed: stove not on")
            self.set_feature(f"{item}_cooked", 1)

        elif action_str.startswith("cook_contents"):
            parts     = _parse(action_str)
            container = parts[0]
            location  = parts[1] if len(parts) > 1 else "cooking_station"
            if self.get_feature(f"{container}_at_{location}") != 1:
                raise ValueError(f"Precondition failed: {container} not at {location}")
            if location == "cooking_station" and self.get_feature("stove_on") != 1:
                raise ValueError("Precondition failed: stove not on")
            for ingredient in INGREDIENTS:
                if self.get_feature(f"{container}_contains_{ingredient}") == 1:
                    if ingredient in COOKABLES:
                        self.set_feature(f"{ingredient}_cooked", 1)

        elif action_str.startswith("combine"):
            parts     = _parse(action_str)
            container = parts[0]
            location  = parts[1] if len(parts) > 1 else None
            if location and self.get_feature(f"{container}_at_{location}") != 1:
                raise ValueError(f"Precondition failed: {container} not at {location}")
            contained = [ing for ing in INGREDIENTS
                         if ing != "mixture" and self.get_feature(f"{container}_contains_{ing}") == 1]
            if len(contained) < 2:
                raise ValueError(f"combine requires >=2 ingredients in {container}")
            for ing in contained:
                self.set_feature(f"{container}_contains_{ing}", 0)
                self.set_feature(f"{ing}_in_mixture", 1)
            self.set_feature(f"{container}_contains_mixture", 1)
            if location:
                self.set_feature(f"mixture_at_{location}", 1)

        elif action_str.startswith("season_container"):
            parts     = _parse(action_str)
            container = parts[0]; seasoning = parts[1]
            location  = parts[2] if len(parts) > 2 else None
            if seasoning not in SEASONINGS:
                raise ValueError(f"{seasoning} is not a seasoning")
            if location and self.get_feature(f"{container}_at_{location}") != 1:
                raise ValueError(f"{container} not at {location}")
            for ing in INGREDIENTS:
                if self.get_feature(f"{container}_contains_{ing}") == 1:
                    self.set_feature(f"{ing}_seasoned", 1)
                    self.set_feature(f"{ing}_seasoned_with_{seasoning}", 1)

        elif action_str.startswith("season"):
            parts     = _parse(action_str)
            target    = parts[0]; seasoning = parts[1]
            location  = parts[2] if len(parts) > 2 else None
            if seasoning not in SEASONINGS:
                raise ValueError(f"{seasoning} is not a seasoning")
            if location and self.get_feature(f"{target}_at_{location}") != 1:
                raise ValueError(f"{target} not at {location}")
            self.set_feature(f"{target}_seasoned", 1)
            self.set_feature(f"{target}_seasoned_with_{seasoning}", 1)

        elif action_str.startswith("pour"):
            parts          = _parse(action_str)
            liquid         = parts[0]; from_c = parts[1]; to_c = parts[2]
            location       = parts[3] if len(parts) > 3 else None
            if liquid not in LIQUID_INGREDIENTS:
                raise ValueError(f"{liquid} is not a liquid")
            if self.get_feature(f"{from_c}_contains_{liquid}") != 1:
                raise ValueError(f"{liquid} not in {from_c}")
            if location:
                if self.get_feature(f"{from_c}_at_{location}") != 1:
                    raise ValueError(f"{from_c} not at {location}")
                if self.get_feature(f"{to_c}_at_{location}") != 1:
                    raise ValueError(f"{to_c} not at {location}")
            self.set_feature(f"{from_c}_contains_{liquid}", 0)
            self.set_feature(f"{to_c}_contains_{liquid}", 1)
            if location:
                self.set_feature(f"{liquid}_at_{location}", 1)

        elif action_str.startswith("turn_on"):
            tool = _parse(action_str)[0]
            if   tool == "stove":   self.set_feature("stove_on",   1)
            elif tool == "sink":    self.set_feature("sink_on",    1)
            elif tool == "blender": self.set_feature("blender_on", 1)

        elif action_str.startswith("turn_off"):
            tool = _parse(action_str)[0]
            if   tool == "stove":   self.set_feature("stove_on",   0)
            elif tool == "sink":    self.set_feature("sink_on",    0)
            elif tool == "blender": self.set_feature("blender_on", 0)

        elif action_str.startswith("blend"):
            parts     = _parse(action_str)
            container = parts[0]
            location  = parts[1] if len(parts) > 1 else "blending_station"
            if self.get_feature(f"{container}_at_{location}") != 1:
                raise ValueError(f"{container} not at {location}")
            if self.get_feature("blender_on") != 1:
                raise ValueError("blender not on")
            contained = [ing for ing in INGREDIENTS
                         if ing != "mixture" and self.get_feature(f"{container}_contains_{ing}") == 1]
            if len(contained) < 1:
                raise ValueError(f"blend requires >=1 ingredient in {container}")
            for ing in contained:
                self.set_feature(f"{container}_contains_{ing}", 0)
                self.set_feature(f"{ing}_in_mixture", 1)
            self.set_feature(f"{container}_contains_mixture", 1)
            self.set_feature(f"mixture_at_{location}", 1)

        elif action_str.startswith("serve"):
            parts    = _parse(action_str)
            plate    = parts[0]
            location = parts[1] if len(parts) > 1 else "serving_station"
            if self.get_feature(f"{plate}_at_{location}") != 1:
                raise ValueError(f"{plate} not at {location}")
            self.set_feature("plate_served", 1)

        elif action_str.startswith("wash"):
            parts    = _parse(action_str)
            item     = parts[0]
            location = parts[1] if len(parts) > 1 else "washing_station"
            if self.get_feature(f"{item}_at_{location}") != 1:
                raise ValueError(f"{item} not at {location}")
            self.set_feature(f"{item}_washed", 1)

        else:
            raise ValueError(f"Unknown action: {action_str}")


# Module-level feature map reference (single object, not rebuilt per call)
_FEAT = StateTracker._build_feature_map()

_t = StateTracker()
print(f"{_t.n_features} raw state features")








class RecipeGenerator:
    def __init__(self):
        self.demos   = []
        self.tracker = StateTracker()

    def _record_trajectory(self, actions):
        self.tracker.reset()
        trajectory = []
        for action in actions:
            state_vector = self.tracker.get_state_vector().tolist()
            trajectory.append({"state": state_vector, "action": action})
            try:
                self.tracker.apply_action(action)
            except ValueError as e:
                print(f"ERROR applying action '{action}': {e}")
                raise
        final_state_vector = self.tracker.get_state_vector().tolist()
        trajectory.append({"state": final_state_vector, "action": "stop"})
        self.demos.append(trajectory)

    def generate_grilled_steak(self): return [
        "transfer (pan, from=storage, to=cooking_station)",
        "load (meat, bowl, storage)",
        "move_container (bowl, from=storage, to=cooking_station)",
        "unload (meat, bowl, cooking_station)",
        "load (meat, pan, cooking_station)",
        "turn_on (stove, cooking_station)",
        "cook_contents (pan, cooking_station)",
        "turn_off (stove, cooking_station)",
        "transfer (plate, from=storage, to=plating_station)",
        "unload (meat, pan, cooking_station)",
        "load (meat, bowl, cooking_station)",
        "move_container (bowl, from=cooking_station, to=plating_station)",
        "unload (meat, bowl, plating_station)",
        "load (meat, plate, plating_station)",
        "move_container (plate, from=plating_station, to=serving_station)",
        "serve (plate, serving_station)",
        "transfer (pan, from=cooking_station, to=washing_station)",
        "wash (pan, washing_station)",
        "transfer (bowl, from=plating_station, to=washing_station)",
        "wash (bowl, washing_station)"]

    def generate_boiled_eggs(self): return [
        "transfer (pot, from=storage, to=cooking_station)",
        "load (egg, bowl, storage)",
        "move_container (bowl, from=storage, to=cooking_station)",
        "unload (egg, bowl, cooking_station)",
        "load (egg, pot, cooking_station)",
        "turn_on (stove, cooking_station)",
        "cook_contents (pot, cooking_station)",
        "turn_off (stove, cooking_station)",
        "transfer (plate, from=storage, to=plating_station)",
        "move_container (pot, from=cooking_station, to=plating_station)",
        "unload (egg, pot, plating_station)",
        "load (egg, plate, plating_station)",
        "move_container (plate, from=plating_station, to=serving_station)",
        "serve (plate, serving_station)",
        "transfer (pot, from=plating_station, to=washing_station)",
        "wash (pot, washing_station)",
        "transfer (bowl, from=cooking_station, to=washing_station)",
        "wash (bowl, washing_station)"]

    def generate_boiled_rice(self): return [
        "transfer (pot, from=storage, to=cooking_station)",
        "load (rice, bowl, storage)",
        "move_container (bowl, from=storage, to=cooking_station)",
        "unload (rice, bowl, cooking_station)",
        "turn_on (stove, cooking_station)",
        "load (rice, pot, cooking_station)",
        "cook_contents (pot, cooking_station)",
        "turn_off (stove, cooking_station)",
        "transfer (plate, from=storage, to=plating_station)",
        "unload (rice, pot, cooking_station)",
        "load (rice, bowl, cooking_station)",
        "move_container (bowl, from=cooking_station, to=plating_station)",
        "unload (rice, bowl, plating_station)",
        "load (rice, plate, plating_station)",
        "move_container (plate, from=plating_station, to=serving_station)",
        "serve (plate, serving_station)",
        "transfer (pot, from=cooking_station, to=washing_station)",
        "wash (pot, washing_station)",
        "transfer (bowl, from=plating_station, to=washing_station)",
        "wash (bowl, washing_station)"]

    def generate_simple_salad(self): return [
        "transfer (bowl, from=storage, to=prep_station)",
        "transfer (lettuce, from=storage, to=prep_station)",
        "transfer (onion, from=storage, to=prep_station)",
        "cut (lettuce, prep_station)",
        "load (lettuce, bowl, prep_station)",
        "cut (onion, prep_station)",
        "load (onion, bowl, prep_station)",
        "combine (bowl, prep_station)",
        "transfer (plate, from=storage, to=plating_station)",
        "move_container (bowl, from=prep_station, to=plating_station)",
        "unload (mixture, bowl, plating_station)",
        "load (mixture, plate, plating_station)",
        "move_container (plate, from=plating_station, to=serving_station)",
        "serve (plate, serving_station)",
        "move_container (plate, from=serving_station, to=washing_station)",
        "wash (plate, washing_station)",
        "transfer (bowl, from=plating_station, to=washing_station)",
        "wash (bowl, washing_station)"]

    def generate_burger(self): return [
        "transfer (pan, from=storage, to=cooking_station)",
        "load (meat, bowl, storage)",
        "move_container (bowl, from=storage, to=cooking_station)",
        "unload (meat, bowl, cooking_station)",
        "load (meat, pan, cooking_station)",
        "turn_on (stove, cooking_station)",
        "cook_contents (pan, cooking_station)",
        "turn_off (stove, cooking_station)",
        "transfer (plate, from=storage, to=plating_station)",
        "unload (meat, pan, cooking_station)",
        "load (meat, bowl, cooking_station)",
        "move_container (bowl, from=cooking_station, to=plating_station)",
        "unload (meat, bowl, plating_station)",
        "load (meat, plate, plating_station)",
        "transfer (lettuce, from=storage, to=prep_station)",
        "move_container (bowl, from=plating_station, to=prep_station)",
        "cut (lettuce, prep_station)",
        "load (lettuce, bowl, prep_station)",
        "move_container (bowl, from=prep_station, to=plating_station)",
        "unload (lettuce, bowl, plating_station)",
        "load (lettuce, plate, plating_station)",
        "move_container (plate, from=plating_station, to=serving_station)",
        "serve (plate, serving_station)",
        "transfer (pan, from=cooking_station, to=washing_station)",
        "wash (pan, washing_station)",
        "transfer (bowl, from=plating_station, to=washing_station)",
        "wash (bowl, washing_station)"]

    def generate_tomato_soup(self): return [
        "transfer (pot, from=storage, to=cooking_station)",
        "transfer (bowl, from=storage, to=prep_station)",
        "turn_on (stove, cooking_station)",
        "transfer (tomato, from=storage, to=prep_station)",
        "cut (tomato, prep_station)",
        "load (tomato, bowl, prep_station)",
        "move_container (bowl, from=prep_station, to=cooking_station)",
        "unload (tomato, bowl, cooking_station)",
        "load (tomato, pot, cooking_station)",
        "cook_contents (pot, cooking_station)",
        "turn_off (stove, cooking_station)",
        "transfer (plate, from=storage, to=plating_station)",
        "unload (tomato, pot, cooking_station)",
        "load (tomato, bowl, cooking_station)",
        "move_container (bowl, from=cooking_station, to=plating_station)",
        "unload (tomato, bowl, plating_station)",
        "load (tomato, plate, plating_station)",
        "move_container (plate, from=plating_station, to=serving_station)",
        "serve (plate, serving_station)",
        "transfer (pot, from=cooking_station, to=washing_station)",
        "wash (pot, washing_station)",
        "transfer (bowl, from=plating_station, to=washing_station)",
        "wash (bowl, washing_station)"]

    def generate_tomato_onion_soup_v1(self): return [
        "transfer (pot, from=storage, to=cooking_station)",
        "transfer (bowl, from=storage, to=prep_station)",
        "transfer (tomato, from=storage, to=prep_station)",
        "cut (tomato, prep_station)",
        "load (tomato, bowl, prep_station)",
        "transfer (onion, from=storage, to=prep_station)",
        "cut (onion, prep_station)",
        "load (onion, bowl, prep_station)",
        "combine (bowl, prep_station)",
        "move_container (bowl, from=prep_station, to=cooking_station)",
        "unload (mixture, bowl, cooking_station)",
        "load (mixture, pot, cooking_station)",
        "turn_on (stove, cooking_station)",
        "cook_contents (pot, cooking_station)",
        "turn_off (stove, cooking_station)",
        "transfer (plate, from=storage, to=plating_station)",
        "move_container (pot, from=cooking_station, to=plating_station)",
        "unload (mixture, pot, plating_station)",
        "load (mixture, plate, plating_station)",
        "move_container (plate, from=plating_station, to=serving_station)",
        "serve (plate, serving_station)",
        "move_container (pot, from=plating_station, to=washing_station)",
        "wash (pot, washing_station)",
        "transfer (bowl, from=cooking_station, to=washing_station)",
        "wash (bowl, washing_station)"]

    def generate_tomato_onion_soup_v2(self): return [
        "transfer (pot, from=storage, to=cooking_station)",
        "transfer (bowl, from=storage, to=prep_station)",
        "transfer (onion, from=storage, to=prep_station)",
        "cut (onion, prep_station)",
        "load (onion, bowl, prep_station)",
        "transfer (tomato, from=storage, to=prep_station)",
        "cut (tomato, prep_station)",
        "load (tomato, bowl, prep_station)",
        "combine (bowl, prep_station)",
        "move_container (bowl, from=prep_station, to=cooking_station)",
        "unload (mixture, bowl, cooking_station)",
        "load (mixture, pot, cooking_station)",
        "turn_on (stove, cooking_station)",
        "cook_contents (pot, cooking_station)",
        "turn_off (stove, cooking_station)",
        "transfer (plate, from=storage, to=plating_station)",
        "move_container (pot, from=cooking_station, to=plating_station)",
        "unload (mixture, pot, plating_station)",
        "load (mixture, plate, plating_station)",
        "move_container (plate, from=plating_station, to=serving_station)",
        "serve (plate, serving_station)",
        "move_container (pot, from=plating_station, to=washing_station)",
        "wash (pot, washing_station)",
        "transfer (bowl, from=cooking_station, to=washing_station)",
        "wash (bowl, washing_station)"]

    def generate_mushroom_soup(self): return [
        "transfer (pot, from=storage, to=cooking_station)",
        "transfer (bowl, from=storage, to=prep_station)",
        "transfer (mushroom, from=storage, to=prep_station)",
        "cut (mushroom, prep_station)",
        "load (mushroom, bowl, prep_station)",
        "transfer (onion, from=storage, to=prep_station)",
        "cut (onion, prep_station)",
        "load (onion, bowl, prep_station)",
        "combine (bowl, prep_station)",
        "move_container (bowl, from=prep_station, to=cooking_station)",
        "unload (mixture, bowl, cooking_station)",
        "load (mixture, pot, cooking_station)",
        "turn_on (stove, cooking_station)",
        "cook_contents (pot, cooking_station)",
        "turn_off (stove, cooking_station)",
        "transfer (plate, from=storage, to=plating_station)",
        "move_container (pot, from=cooking_station, to=plating_station)",
        "unload (mixture, pot, plating_station)",
        "load (mixture, plate, plating_station)",
        "move_container (plate, from=plating_station, to=serving_station)",
        "serve (plate, serving_station)",
        "move_container (pot, from=plating_station, to=washing_station)",
        "wash (pot, washing_station)",
        "transfer (bowl, from=cooking_station, to=washing_station)",
        "wash (bowl, washing_station)"]

    def generate_seasoned_chicken(self): return [
        "transfer (pan, from=storage, to=cooking_station)",
        "transfer (bowl, from=storage, to=prep_station)",
        "transfer (chicken, from=storage, to=prep_station)",
        "season (chicken, salt, prep_station)",
        "season (chicken, spice1, prep_station)",
        "load (chicken, bowl, prep_station)",
        "move_container (bowl, from=prep_station, to=cooking_station)",
        "unload (chicken, bowl, cooking_station)",
        "load (chicken, pan, cooking_station)",
        "turn_on (stove, cooking_station)",
        "cook_contents (pan, cooking_station)",
        "turn_off (stove, cooking_station)",
        "transfer (plate, from=storage, to=plating_station)",
        "unload (chicken, pan, cooking_station)",
        "load (chicken, bowl, cooking_station)",
        "move_container (bowl, from=cooking_station, to=plating_station)",
        "unload (chicken, bowl, plating_station)",
        "load (chicken, plate, plating_station)",
        "move_container (plate, from=plating_station, to=serving_station)",
        "serve (plate, serving_station)",
        "transfer (pan, from=cooking_station, to=washing_station)",
        "wash (pan, washing_station)",
        "transfer (bowl, from=plating_station, to=washing_station)",
        "wash (bowl, washing_station)"]

    def generate_garlic_fish(self): return [
        "transfer (pan, from=storage, to=cooking_station)",
        "transfer (bowl, from=storage, to=prep_station)",
        "transfer (fish, from=storage, to=prep_station)",
        "season (fish, garlic, prep_station)",
        "season (fish, spice2, prep_station)",
        "load (fish, bowl, prep_station)",
        "move_container (bowl, from=prep_station, to=cooking_station)",
        "unload (fish, bowl, cooking_station)",
        "load (fish, pan, cooking_station)",
        "turn_on (stove, cooking_station)",
        "cook_contents (pan, cooking_station)",
        "turn_off (stove, cooking_station)",
        "transfer (plate, from=storage, to=plating_station)",
        "unload (fish, pan, cooking_station)",
        "load (fish, bowl, cooking_station)",
        "move_container (bowl, from=cooking_station, to=plating_station)",
        "unload (fish, bowl, plating_station)",
        "load (fish, plate, plating_station)",
        "move_container (plate, from=plating_station, to=serving_station)",
        "serve (plate, serving_station)",
        "transfer (pan, from=cooking_station, to=washing_station)",
        "wash (pan, washing_station)",
        "transfer (bowl, from=plating_station, to=washing_station)",
        "wash (bowl, washing_station)"]

    def generate_seasoned_mixture_soup(self): return [
        "transfer (pot, from=storage, to=cooking_station)",
        "transfer (bowl, from=storage, to=prep_station)",
        "transfer (tomato, from=storage, to=prep_station)",
        "cut (tomato, prep_station)",
        "load (tomato, bowl, prep_station)",
        "transfer (onion, from=storage, to=prep_station)",
        "cut (onion, prep_station)",
        "load (onion, bowl, prep_station)",
        "combine (bowl, prep_station)",
        "season_container (bowl, salt, prep_station)",
        "season_container (bowl, spice1, prep_station)",
        "move_container (bowl, from=prep_station, to=cooking_station)",
        "unload (mixture, bowl, cooking_station)",
        "load (mixture, pot, cooking_station)",
        "turn_on (stove, cooking_station)",
        "cook_contents (pot, cooking_station)",
        "turn_off (stove, cooking_station)",
        "transfer (plate, from=storage, to=plating_station)",
        "move_container (pot, from=cooking_station, to=plating_station)",
        "unload (mixture, pot, plating_station)",
        "load (mixture, plate, plating_station)",
        "move_container (plate, from=plating_station, to=serving_station)",
        "serve (plate, serving_station)",
        "move_container (pot, from=plating_station, to=washing_station)",
        "wash (pot, washing_station)",
        "transfer (bowl, from=cooking_station, to=washing_station)",
        "wash (bowl, washing_station)"]

    def generate_grated_cheese_salad(self): return [
        "transfer (bowl, from=storage, to=prep_station)",
        "transfer (lettuce, from=storage, to=prep_station)",
        "transfer (cheese, from=storage, to=prep_station)",
        "cut (lettuce, prep_station)",
        "load (lettuce, bowl, prep_station)",
        "grate (cheese, prep_station)",
        "load (cheese, bowl, prep_station)",
        "combine (bowl, prep_station)",
        "transfer (plate, from=storage, to=plating_station)",
        "move_container (bowl, from=prep_station, to=plating_station)",
        "unload (mixture, bowl, plating_station)",
        "load (mixture, plate, plating_station)",
        "move_container (plate, from=plating_station, to=serving_station)",
        "serve (plate, serving_station)",
        "move_container (plate, from=serving_station, to=washing_station)",
        "wash (plate, washing_station)",
        "transfer (bowl, from=plating_station, to=washing_station)",
        "wash (bowl, washing_station)"]

    def generate_smoothie(self): return [
        "transfer (glass, from=storage, to=blending_station)",
        "transfer (measuring_cup, from=storage, to=blending_station)",
        "transfer (banana, from=storage, to=prep_station)",
        "transfer (strawberries, from=storage, to=prep_station)",
        "cut (banana, prep_station)",
        "transfer (banana, from=prep_station, to=blending_station)",
        "cut (strawberries, prep_station)",
        "transfer (strawberries, from=prep_station, to=blending_station)",
        "transfer (milk, from=storage, to=blending_station)",
        "load (milk, measuring_cup, blending_station)",
        "pour (milk, measuring_cup, glass, blending_station)",
        "load (banana, glass, blending_station)",
        "load (strawberries, glass, blending_station)",
        "turn_on (blender, blending_station)",
        "blend (glass, blending_station)",
        "turn_off (blender, blending_station)",
        "transfer (glass, from=blending_station, to=serving_station)",
        "serve (glass, serving_station)",
        "transfer (measuring_cup, from=blending_station, to=washing_station)",
        "wash (measuring_cup, washing_station)"]

    def generate_yoghurt_smoothie(self): return [
        "transfer (glass, from=storage, to=blending_station)",
        "transfer (measuring_cup, from=storage, to=blending_station)",
        "transfer (banana, from=storage, to=prep_station)",
        "transfer (yoghurt, from=storage, to=blending_station)",
        "cut (banana, prep_station)",
        "transfer (banana, from=prep_station, to=blending_station)",
        "load (yoghurt, glass, blending_station)",
        "transfer (milk, from=storage, to=blending_station)",
        "load (milk, measuring_cup, blending_station)",
        "pour (milk, measuring_cup, glass, blending_station)",
        "load (banana, glass, blending_station)",
        "turn_on (blender, blending_station)",
        "blend (glass, blending_station)",
        "turn_off (blender, blending_station)",
        "transfer (glass, from=blending_station, to=serving_station)",
        "serve (glass, serving_station)",
        "transfer (measuring_cup, from=blending_station, to=washing_station)",
        "wash (measuring_cup, washing_station)"]

    def generate_random_dataset(self, count):
        """Generate count demonstrations drawn uniformly from ALL available recipes."""
        available_recipes = [
            self.generate_tomato_onion_soup_v1, self.generate_tomato_onion_soup_v2,
            self.generate_tomato_soup,          self.generate_mushroom_soup,
            self.generate_seasoned_mixture_soup, self.generate_grilled_steak,
            self.generate_burger,               self.generate_seasoned_chicken,
            self.generate_garlic_fish,          self.generate_simple_salad,
            self.generate_grated_cheese_salad,  self.generate_smoothie,
            self.generate_yoghurt_smoothie,     self.generate_boiled_eggs,
            self.generate_boiled_rice,
        ]
        print(f"Generating {count} demonstrations from {len(available_recipes)} recipes...")
        for _ in range(count):
            recipe_func = random.choice(available_recipes)
            actions     = recipe_func()
            self._record_trajectory(actions)

    def save_to_file(self, filepath="demonstrations.txt"):
        with open(filepath, "w") as f:
            for demo in self.demos:
                f.write(json.dumps(demo) + "\n")
        print(f"Saved {len(self.demos)} trajectories → {filepath}")

gen = RecipeGenerator()
print(f"{len([m for m in dir(gen) if m.startswith('generate_')])} recipes defined")

470 raw state features
16 recipes defined


## Cell 2: Feature Engineering, IRL, Demo-Augmented MDP MaxEnt

In [ ]:
#  _safe_feat helper eliminates ~120 lines of repeated key-lookup patterns. create_enhanced_feature_matrix is the single source of engineered features; its output dimensionality is FIXED by the constant vocabulary lists, so warm-starting across retrains is always valid.

def _safe_feat(state, key):
    """Return state[_FEAT[key]] if key exists, else 0. Avoids key-error boilerplate."""
    return int(state[_FEAT[key]]) if key in _FEAT else 0

def create_state_action_mappings(demonstrations, unique_actions):
    """Bidirectional mappings: state ↔ index, action ↔ index."""
    state_to_idx = {}
    for trajectory in demonstrations:
        for state, _ in trajectory:
            if state not in state_to_idx:
                state_to_idx[state] = len(state_to_idx)
    idx_to_state  = {idx: state for state, idx in state_to_idx.items()}
    action_to_idx = {action: idx for idx, action in enumerate(unique_actions)}
    idx_to_action = {idx: action for action, idx in action_to_idx.items()}
    return state_to_idx, idx_to_state, action_to_idx, idx_to_action

def create_enhanced_feature_matrix(idx_to_state, col_min_known=None, col_max_known=None):
    """
    Task-discriminative feature matrix from PDDL kitchen states. All raw-state column accesses use the module-level _FEAT dict. The engineered feature dimension is determined entirely by the fixed vocabulary constants below, not by data size, so it is stable across every retrain cycle and warm-starting is always valid.
    Parameters:
    idx_to_state   : {int: state_tuple}
    col_min_known  : normalisation lower bounds from a prior training call
    col_max_known  : normalisation upper bounds (pass for test states)
    Returns:    feature_matrix, col_min, col_max
    """
    KEY_INGREDIENTS = [
        "tomato", "onion", "mushroom", "rice", "meat",
        "chicken", "fish", "egg", "banana", "strawberries",
        "lettuce", "cheese", "garlic", "yoghurt", "milk",
    ]
    KEY_CONTAINERS = ["pot", "pan", "plate", "bowl"]
    KEY_LOCS       = ["prep_station", "cooking_station", "plating_station", "blending_station"]

    features = []
    for idx in range(len(idx_to_state)):
        state = np.array(idx_to_state[idx])
        feat  = []

        # 1. Items per location (7 features)
        for loc in LOCATIONS:
            feat.append(sum(_safe_feat(state, f"{item}_at_{loc}") for item in ITEMS))

        # 2. Container × location presence (16 + 3 composite)
        for container, loc in [
            ("pot",          "cooking_station"),  ("pan",           "cooking_station"),
            ("plate",        "plating_station"),  ("plate",         "serving_station"),
            ("plate",        "washing_station"),  ("bowl",          "prep_station"),
            ("bowl",         "cooking_station"),  ("bowl",          "plating_station"),
            ("glass",        "blending_station"), ("measuring_cup", "blending_station"),
            ("glass",        "serving_station"),  ("pot",           "washing_station"),
            ("pan",          "washing_station"),  ("bowl",          "washing_station"),
            ("glass",        "washing_station"),  ("measuring_cup", "washing_station"),
        ]:
            feat.append(_safe_feat(state, f"{container}_at_{loc}"))
        cooking_vessel = (_safe_feat(state, "pot_at_cooking_station") + _safe_feat(state, "pan_at_cooking_station"))
        feat.append(cooking_vessel)
        feat.append(sum(_safe_feat(state, f"{item}_at_prep_station")     for item in ITEMS))
        feat.append(sum(_safe_feat(state, f"{item}_at_blending_station") for item in ITEMS))

        # 3. Containment aggregates (8 features)
        total_contained = sum(_safe_feat(state, f"{c}_contains_{i}") for c in CONTAINERS for i in INGREDIENTS)
        feat.append(total_contained)
        for container in CONTAINERS: feat.append(sum(_safe_feat(state, f"{container}_contains_{ing}") for ing in INGREDIENTS))
        feat.append(sum(_safe_feat(state, f"{c}_contains_mixture") for c in CONTAINERS))

        # 4. Processing-state features
        cut_vals    = [_safe_feat(state, f"{item}_cut")    for item in CUTTABLES]
        cooked_vals = [_safe_feat(state, f"{item}_cooked") for item in COOKABLES]
        feat.extend(cut_vals)
        feat.extend(cooked_vals)
        total_cut      = sum(cut_vals)
        total_cooked   = sum(cooked_vals)
        total_grated   = sum(_safe_feat(state, f"{item}_grated")   for item in GRATABLE)
        total_seasoned = sum(_safe_feat(state, f"{item}_seasoned")  for item in INGREDIENTS)
        total_washed   = sum(_safe_feat(state, f"{item}_washed")    for item in ITEMS)
        feat.extend([total_cut, total_grated, total_cooked, total_seasoned])
        for ing in ["chicken", "fish", "mixture"]: feat.append(_safe_feat(state, f"{ing}_seasoned"))
        feat.append(total_washed)

        # 5. Tool-usage features (6 features)
        stove_on   = _safe_feat(state, "stove_on")
        sink_on    = _safe_feat(state, "sink_on")
        blender_on = _safe_feat(state, "blender_on")
        items_in_glass = sum(_safe_feat(state, f"glass_contains_{ing}") for ing in INGREDIENTS)
        feat.extend([stove_on,      stove_on * cooking_vessel,      stove_on * total_contained,         sink_on, blender_on,        blender_on * items_in_glass])

        # 6. Workflow-progress features (11 features)
        served        = _safe_feat(state, "plate_served")
        items_in_plate = sum(_safe_feat(state, f"plate_contains_{ing}") for ing in INGREDIENTS)
        feat.extend([int(total_cut > 0), int(total_cooked > 0), int(items_in_plate > 0), int(items_in_glass > 0), served, int(total_washed > 0), int(total_grated > 0), int(total_seasoned > 0), int(total_cut > 0) * int(total_cooked > 0) * served * int(total_washed > 0), int(total_cut > 0) * served * int(total_washed > 0) * int(total_cooked == 0), served * int(total_washed > 0)])

        # 7-11. Discriminative features (ingredient × location/container/seasoning)
        for ing in KEY_INGREDIENTS:
            for loc in KEY_LOCS:
                feat.append(_safe_feat(state, f"{ing}_at_{loc}"))
        for ing in KEY_INGREDIENTS:
            for cont in KEY_CONTAINERS:
                feat.append(_safe_feat(state, f"{cont}_contains_{ing}"))
        for ing in KEY_INGREDIENTS:
            feat.append(_safe_feat(state, f"{ing}_seasoned"))
        for ing in KEY_INGREDIENTS:
            if ing in INGREDIENTS:
                feat.append(_safe_feat(state, f"{ing}_in_mixture"))
        for seasoning in SEASONINGS:
            for ing in KEY_INGREDIENTS:
                if ing in INGREDIENTS:
                    feat.append(_safe_feat(state, f"{ing}_seasoned_with_{seasoning}"))
        features.append(feat)

    feature_matrix = np.array(features, dtype=float)
    col_min   = col_min_known if col_min_known is not None else feature_matrix.min(axis=0)
    col_max   = col_max_known if col_max_known is not None else feature_matrix.max(axis=0)
    col_range = col_max - col_min
    col_range[col_range == 0] = 1.0
    feature_matrix = np.clip((feature_matrix - col_min) / col_range, 0.0, 1.0)
    return feature_matrix, col_min, col_max

_test_fm, _, _ = create_enhanced_feature_matrix({0: tuple(StateTracker().get_state_vector().tolist())})
print(f"✓ Feature engineering ready — engineered feature dim: {_test_fm.shape[1]} (fixed)")
# ═══════════════════════════════════════════════════════════════════════════════
#  IMPORTANCE-WEIGHTED MaxEnt IRL  —  Demo-Augmented MDP
#
#  Theoretical framing
#  ───────────────────
#  Standard MaxEnt IRL (Ziebart et al., 2008) requires expected feature counts under the learned policy over the FULL MDP.  With demo-only transitions, the expected counts equal the empirical counts → gradient ≈ 0 → no learning.
#
#  Fix: Demo-Augmented MDP
#  For each demonstrated state s, we add one virtual DEVIATE action whose transition leads to a null state (zero-feature absorbing state).  The soft Bellman VI must now assign Q(s, deviate) < Q(s, demonstrated) for the policy to prefer demonstrated actions.  
#  The expected feature counts under the learned policy can now differ from the empirical counts, making the IRL gradient non-trivial.  Demonstrated actions are rewarded; the deviate option is penalised by receiving zero features.
#  This is equivalent to "maximum entropy IRL on a demo-constrained MDP with a null alternative" — see Bloem & Bambos (2014) for theoretical grounding.
#
#  Importance weighting
#  ───────────────────────────────────────────────
#  θ* = argmax_θ  Σ_i w_i · log P(τ_i | θ) where w_i are the time-decayed demo weights from AdaptiveIRL. Recent demos contribute more to the gradient than stale ones.
#
#  Warm-starting validity
#  ──────────────────────
#  reward_weights ∈ R^n_features where n_features is determined by the FIXED vocabulary constants in create_enhanced_feature_matrix, NOT by data size. Adding new recipes only enlarges state/action index spaces, never n_features. Warm-starting is therefore always valid across every retrain cycle.
# ═══════════════════════════════════════════════════════════════════════════════

def max_ent_irl(demonstrations, feature_matrix, state_to_idx, action_to_idx, n_iterations=100, learning_rate=0.05, temperature=0.5, gamma=0.9, init_weights=None, demo_weights=None):
    n_states, n_features = feature_matrix.shape
    n_actions = len(action_to_idx)
    MC_Count  = min(max(150, 10 * len(demonstrations)), 400)
    verbose   = True


    # Normalise demo weights 
    w = (np.ones(len(demonstrations)) if demo_weights is None else np.array(demo_weights, dtype=float))
    w_sum = w.sum()
    if w_sum == 0: w, w_sum = np.ones(len(demonstrations)), float(len(demonstrations))
    demo_probs = w / w_sum

    if verbose: print(f"  MaxEnt IRL — States: {n_states}, Actions: {n_actions}, Features: {n_features}, MC rollouts: {MC_Count}")
    if verbose and demo_weights is not None:
        entropy     = float(-(demo_probs * np.log(demo_probs + 1e-12)).sum())
        max_entropy = float(np.log(max(len(demonstrations), 1)))
        uniformity  = entropy / max_entropy if max_entropy > 0 else 1.0
        print(f"  IW-MaxEnt weight-entropy={entropy:.3f}/{max_entropy:.3f}  uniformity={uniformity:.2f} (1.0=uniform, <1.0=recency-biased)")


    # Reward weights init
    if init_weights is not None and len(init_weights) == n_features:
        reward_weights = init_weights.copy()
        if verbose: print(f"  Warm-start ‖w‖={np.linalg.norm(init_weights):.3f}")
    else:
        reward_weights = np.random.randn(n_features) * 0.1
        if verbose: print(f"  Random init (first cycle or dim mismatch)")


    # Build base transition model and s_to_actions
    state_action_pairs = set()
    transition_model   = {}
    s_to_actions       = {}

    for trajectory in demonstrations:
        for i in range(len(trajectory)):
            state, action = trajectory[i]
            s_idx = state_to_idx[state]
            a_idx = action_to_idx[action]
            state_action_pairs.add((s_idx, a_idx))
            s_to_actions.setdefault(s_idx, [])
            if a_idx not in s_to_actions[s_idx]:    s_to_actions[s_idx].append(a_idx)
            if i < len(trajectory) - 1:             transition_model[(s_idx, a_idx)] = state_to_idx[trajectory[i+1][0]]


    # Demo-Augmented MDP
    # Add one DEVIATE action per demonstrated state → null state. The null state has a zero feature vector and no outgoing transitions. This gives the soft Bellman VI a non-trivial contrast at every state, making the expected feature counts under the policy differ from empirical.
    DEVIATE_A     = n_actions        # virtual action index
    NULL_S        = n_states         # virtual null state index
    null_row      = np.zeros((1, n_features), dtype=float)
    feat_aug      = np.vstack([feature_matrix, null_row])   # (n_states+1, n_features)
    n_states_aug  = n_states + 1

    for s_idx in list(s_to_actions.keys()):
        transition_model[(s_idx, DEVIATE_A)] = NULL_S
        s_to_actions[s_idx] = s_to_actions[s_idx] + [DEVIATE_A]

    # Weighted empirical feature expectations
    empirical_fe = np.zeros(n_features)
    demo_starts  = [state_to_idx[demonstrations[i][0][0]] for i in range(len(demonstrations))]
    for i, trajectory in enumerate(demonstrations):
        traj_fe = np.zeros(n_features)
        for t, (state, _) in enumerate(trajectory):
            s_idx       = state_to_idx[state]
            traj_fe     += (gamma ** t) * feature_matrix[s_idx]   # original feature_matrix (no null)
        empirical_fe    += w[i] * traj_fe
    empirical_fe /= w_sum

    if verbose: print(f"  {len(state_action_pairs)} demo (s,a) pairs  ‖μ_emp‖={np.linalg.norm(empirical_fe):.4f}  null-state augmentation active")


    # Gradient ascent with soft Bellman VI on augmented MDP
    rewards  = feat_aug @ reward_weights      # (n_states_aug,)
    values   = rewards.copy()
    q_values = {}
    best_diff, best_weights = float('inf'), reward_weights.copy()
    momentum, momentum_beta = np.zeros(n_features), 0.9
    no_improve = 0

    for iteration in range(n_iterations):
        lr      = learning_rate * (0.97 ** (no_improve // 20))
        rewards = feat_aug @ reward_weights

        # Soft Bellman VI (augmented)
        for _ in range(50):
            new_values = rewards.copy()
            for (s_idx, a_idx) in state_action_pairs:
                ns = transition_model.get((s_idx, a_idx))
                q_values[(s_idx, a_idx)] = rewards[s_idx] + (gamma * values[ns] if ns is not None else 0.0)
            # Deviate transitions
            for s_idx in s_to_actions:
                if (s_idx, DEVIATE_A) not in q_values: q_values[(s_idx, DEVIATE_A)] = rewards[s_idx] + gamma * values[NULL_S]
            for s_idx, a_list in s_to_actions.items():
                avail = np.array([q_values.get((s_idx, a), rewards[s_idx]) for a in a_list])
                max_q = avail.max()
                new_values[s_idx] = (max_q + temperature * np.log(np.sum(np.exp((avail - max_q) / temperature))))
            delta  = np.max(np.abs(new_values - values))
            values = new_values
            if delta < 1e-6: break

        # Soft policy
        policy_table = {}
        for s_idx, a_list in s_to_actions.items():
            sq    = np.array([q_values.get((s_idx, a), rewards[s_idx]) for a in a_list])
            max_q = sq.max()
            probs = np.exp((sq - max_q) / temperature)
            probs /= probs.sum()
            policy_table[s_idx] = (a_list, probs)

        # Weighted MC rollouts (deviate action → terminates rollout)
        expected_fc = np.zeros(n_features)
        for _ in range(MC_Count):
            demo_idx  = np.random.choice(len(demonstrations), p=demo_probs)
            s_idx     = demo_starts[demo_idx]
            traj_feat = np.zeros(n_features)
            for t in range(40):
                if s_idx < n_states:                                traj_feat += (gamma ** t) * feature_matrix[s_idx]   # real state: add features
                if s_idx not in policy_table:                       break
                a_list, probs   = policy_table[s_idx]
                a_idx           = a_list[int(np.random.choice(len(a_list), p=probs))]
                if a_idx == DEVIATE_A:                              break                                               # deviate → null → stop
                if (s_idx, a_idx) not in transition_model:          break
                s_idx = transition_model[(s_idx, a_idx)]
            expected_fc += traj_feat
        expected_fc /= MC_Count

        gradient  = empirical_fe - expected_fc
        gradient -= 0.01 * reward_weights
        grad_norm = np.linalg.norm(gradient)
        momentum  = momentum_beta * momentum + (1 - momentum_beta) * gradient
        reward_weights += lr * momentum

        if grad_norm < best_diff:   best_diff, best_weights, no_improve = grad_norm, reward_weights.copy(), 0
        else:                       no_improve += 1
        if iteration % 10 == 0 and verbose:                         print(f"  iter {iteration:>3}: ‖∇‖={grad_norm:.5f}  reward∈[{reward_weights.min():.2f},{reward_weights.max():.2f}]  best={best_diff:.5f}  lr={lr:.4f}")
        if no_improve > 60:                                         break
    if verbose:                                                     print(f"  Final best ‖∇‖={best_diff:.6f}")

    # ── Final VI on best weights → Q-table for prediction ───────────────────
    best_rewards = feat_aug @ best_weights
    best_values  = best_rewards.copy()
    best_q       = {}
    for _ in range(40):
        new_vals = best_rewards.copy()
        for (s_idx, a_idx) in state_action_pairs:
            ns = transition_model.get((s_idx, a_idx))
            best_q[(s_idx, a_idx)] = best_rewards[s_idx] + (gamma * best_values[ns] if ns is not None else best_rewards[s_idx])
        for s_idx, a_list in s_to_actions.items():
            avail = np.array([best_q.get((s_idx, a), best_rewards[s_idx]) for a in a_list])
            max_q = avail.max()
            new_vals[s_idx] = (max_q + temperature * np.log(np.sum(np.exp((avail - max_q) / temperature))))
        if np.max(np.abs(new_vals - best_values)) < 1e-6:           break
        best_values = new_vals

    # Clean s_to_actions: remove DEVIATE_A (training artifact, not a real action)
    clean_s_to_actions = {s: [a for a in a_list if a != DEVIATE_A] for s, a_list in s_to_actions.items()}
    # Clean best_q: remove DEVIATE_A entries
    best_q_clean = {(s, a): v for (s, a), v in best_q.items() if a != DEVIATE_A}

    return best_weights, best_rewards[:n_states], best_values[:n_states], best_q_clean, clean_s_to_actions



## Predict the most likely next action from a given state.         O(|A_s|) lookup using stored s_to_actions (not O(|A|) scan).        For unseen states: nearest-neighbour cosine lookup in feature space.
def predict_action(state, reward_weights, feature_matrix, state_to_idx, action_to_idx, idx_to_action, transition_model, col_min=None, col_max=None, temperature=0.5, gamma=0.9, values=None, q_table=None, s_to_actions=None):
    if state not in state_to_idx:
        tmp_fm, _, _ = create_enhanced_feature_matrix({0: tuple(state)}, col_min_known=col_min, col_max_known=col_max)
        query  = tmp_fm[0].astype(np.float32)
        norms  = np.linalg.norm(feature_matrix, axis=1)
        q_norm = float(np.linalg.norm(query))
        s_idx  = (int(np.argmax(feature_matrix @ query / (norms * q_norm + 1e-10))) if q_norm > 0 and norms.any() else 0)
    else: s_idx = state_to_idx[state]

    # O(|A_s|) using stored s_to_actions
    if q_table is not None and s_to_actions is not None and s_idx in s_to_actions:      candidates = [(a, q_table[(s_idx, a)]) for a in s_to_actions[s_idx] if (s_idx, a) in q_table]
    elif q_table is not None:                                                           candidates = [(a, q_table[(s_idx, a)]) for a in range(len(action_to_idx)) if (s_idx, a) in q_table]
    else:
        rewards = feature_matrix @ reward_weights
        candidates = [(a, float(rewards[s_idx]) + gamma * float(rewards[transition_model[(s_idx, a)]])) for a in range(len(action_to_idx)) if (s_idx, a) in transition_model]
    if not candidates: return None, {}

    valid_actions, q_vals = zip(*candidates)
    q_arr  = np.array(q_vals, dtype=np.float32)
    max_q  = q_arr.max()
    probs  = np.exp((q_arr - max_q) / temperature)
    probs /= probs.sum()

    return (idx_to_action[valid_actions[int(np.argmax(probs))]],{idx_to_action[a]: float(p) for a, p in zip(valid_actions, probs)})

✓ Feature engineering ready — engineered feature dim: 287 (fixed)
